In [5]:
import pandas as pd
from riot_api_wrapper import *
import json

# Change this to whoever you want
summoner_name = "JustSpoon"
summoner_tag = "MID"  # The part after the # in Doublelift#NA1
region = 'na1'
# Step 1: Get summoner info
me = get_summoner(summoner_name, summoner_tag, region)
print(f"Level: {me['summonerLevel']} | PUUID: {me['puuid'][:15]}...")

# Step 2: Get last 20 ranked solo/duo games
match_ids = get_match_history(me['puuid'], count=20)

# Step 3: Pull full match data + extract what you care about
matches = []
for match_id in match_ids:
    data = get_match_details(match_id)
    if data:
        info = data['info']
        for participant in info['participants']:
            if participant['puuid'] == me['puuid']:
                matches.append({
                    'gameMode': info['gameMode'],
                    'champion': participant['championName'],
                    'kills': participant['kills'],
                    'deaths': participant['deaths'],
                    'assists': participant['assists'],
                    'win': participant['win'],
                    'kda': round((participant['kills'] + participant['assists']) / max(participant['deaths'], 1), 2),
                    'gameDuration': info['gameDuration'] // 60,
                    'gameEndTimestamp': pd.to_datetime(info['gameEndTimestamp'], unit='ms')
                })
    time.sleep(1)  # be nice to Riot's servers

# Step 4: Turn into beautiful DataFrame
df = pd.DataFrame(matches)
df = df.sort_values('gameEndTimestamp', ascending=False)

# BEST PART – instant insights
print(f"\n{summoner_name}'s last {len(df)} games:")
print(df['win'].value_counts())

display(df.head(10))

Level: 597 | PUUID: SI4MLIq8ZT4oH6t...

JustSpoon's last 20 games:
win
True     11
False     9
Name: count, dtype: int64


,gameMode,champion,kills,deaths,assists,win,kda,gameDuration,gameEndTimestamp
0,CLASSIC,Kayle,19,8,13,True,4.00,38,2026-03-11 03:25:45.636
1,CLASSIC,Kayle,4,5,2,False,1.20,24,2026-03-11 02:35:18.794
2,CLASSIC,Veigar,4,1,3,False,7.00,24,2026-03-11 02:05:30.790
3,CLASSIC,Ahri,4,2,1,True,2.50,26,2026-03-10 21:18:30.827
4,CLASSIC,Kayle,8,9,10,True,2.00,41,2026-03-10 01:56:20.354
5,CLASSIC,Corki,2,5,5,False,1.40,25,2026-03-10 01:03:36.666
6,CLASSIC,Smolder,4,1,5,True,9.00,28,2026-03-10 00:24:23.427
7,CLASSIC,Nasus,0,6,1,False,0.17,24,2026-03-09 23:49:21.805
8,CLASSIC,Veigar,6,4,5,False,2.75,35,2026-03-09 23:19:05.386
9,CLASSIC,Nasus,3,9,7,False,1.11,32,2026-03-08 06:09:16.758
